In [2]:
import json, os
from pathlib import Path

def yolo_norm_to_coco_xywh(cx, cy, w, h, img_w, img_h):
    """Convert normalized YOLO center boxes to COCO xywh (absolute px)."""
    x = (cx - w/2.0) * img_w
    y = (cy - h/2.0) * img_h
    w = w * img_w
    h = h * img_h
    return [float(x), float(y), float(w), float(h)]

def build_coco_from_val_folder(val_dir, out_json):
    val_dir = Path(val_dir)
    json_files = sorted(val_dir.glob("*.json"))

    coco = {
        "info": {"description": "Symbols grouped by key", "version": "1.0"},
        "licenses": [{"id": 1, "name": "private"}],
        "images": [],
        "annotations": [],
        "categories": []
    }

    # category lookup: key -> id
    cat2id = {}
    next_cat_id = 1

    next_image_id = 1
    next_anno_id = 1

    for jf in json_files:
        with open(jf, "r", encoding="utf-8") as f:
            data = json.load(f)

        file_name = data.get("image") or (Path(jf).with_suffix(".png").name)
        img_w = int(data["image_width"])
        img_h = int(data["image_height"])

        coco["images"].append({
            "id": next_image_id,
            "file_name": file_name,
            "width": img_w,
            "height": img_h
        })

        for box in data.get("boxes", []):
            if box.get("cat") != "symbol":
                continue

            key = box.get("key", "unknown")

            # assign category_id for this key
            if key not in cat2id:
                cat2id[key] = next_cat_id
                coco["categories"].append({
                    "id": next_cat_id,
                    "name": key,
                    "supercategory": "symbol"
                })
                next_cat_id += 1

            cx = float(box["norm_x_center"])
            cy = float(box["norm_y_center"])
            bw = float(box["norm_width"])
            bh = float(box["norm_height"])
            xywh = yolo_norm_to_coco_xywh(cx, cy, bw, bh, img_w, img_h)
            area = xywh[2] * xywh[3]

            coco["annotations"].append({
                "id": next_anno_id,
                "image_id": next_image_id,
                "category_id": cat2id[key],
                "bbox": [round(v, 2) for v in xywh],
                "area": round(area, 2),
                "iscrowd": 0
            })
            next_anno_id += 1

        next_image_id += 1

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(coco, f, indent=2)
    print(f"Wrote COCO annotations to {out_json}")

if __name__ == "__main__":
    VAL_DIR = "/Users/linkaiyuan/文件/github/IMG2SCH/data/labels/8_17_4"
    OUT_JSON = "/Users/linkaiyuan/文件/github/IMG2SCH/data/COCO_Format_dataset/dataset/annotations/instances_val.json"
    build_coco_from_val_folder(VAL_DIR, OUT_JSON)


Wrote COCO annotations to /Users/linkaiyuan/文件/github/IMG2SCH/data/COCO_Format_dataset/dataset/annotations/instances_val.json
